# Experiment 7: Random Forests — OOB Error, Bagging, and Feature Subsampling

**Dataset:** Placement Dataset  
**Target:** `PlacementStatus`

### Aim
To study Random Forests by:
1. Measuring Out-of-Bag (OOB) error.
2. Comparing Bagging with Random Forests.
3. Studying the effect of feature subsampling.
4. Observing validation accuracy and feature importance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


## 1. Load the Dataset

The same placement dataset used in the previous experiment is used here.  
The target variable is `PlacementStatus`.


In [ ]:
# If running in Google Colab, upload the CSV file
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
print(df.head())


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nPlacementStatus distribution:")
print(df["PlacementStatus"].value_counts())


## 2. Separate Features and Target

`PlacementStatus` is the target variable. All other columns are used as input features.


In [ ]:
target = "PlacementStatus"

X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 3. Encode Categorical Features

Categorical columns are converted into numerical dummy variables so that tree-based models can use them.


In [ ]:
X = pd.get_dummies(X, drop_first=True)

# Replace infinite values with missing values
X = X.replace([np.inf, -np.inf], np.nan)

print("Encoded X shape:", X.shape)
print("Total missing values:", X.isna().sum().sum())


## 4. Train–Validation Split and Imputation

The data is split into 80% training and 20% validation data.  
Median imputation is fitted only on the training data and then applied to both sets.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)

print("X_train:", X_train_imp.shape)
print("X_val:", X_val_imp.shape)


## 5. Random Forest and OOB Error

Random Forest trains many decision trees using bootstrap samples and random subsets of features.

For each tree, some training observations are not selected in its bootstrap sample. These are called **Out-of-Bag (OOB)** observations. The OOB score provides an internal estimate of generalization performance.

**OOB Error = 1 − OOB Score**


In [ ]:
n_trees_values = [10, 25, 50, 100, 200, 300]

rf_results = []

for n in n_trees_values:
    rf = RandomForestClassifier(
        n_estimators=n,
        max_features="sqrt",
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_imp, y_train)

    val_pred = rf.predict(X_val_imp)
    val_acc = accuracy_score(y_val, val_pred)

    rf_results.append({
        "n_estimators": n,
        "OOB_Score": rf.oob_score_,
        "OOB_Error": 1 - rf.oob_score_,
        "Validation_Accuracy": val_acc
    })

rf_results_df = pd.DataFrame(rf_results)
rf_results_df


In [ ]:
best_rf_row = rf_results_df.loc[
    rf_results_df["Validation_Accuracy"].idxmax()
]

print("Best Random Forest configuration:")
print(best_rf_row)


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    rf_results_df["n_estimators"],
    rf_results_df["OOB_Error"],
    marker="o",
    label="OOB Error"
)

plt.plot(
    rf_results_df["n_estimators"],
    rf_results_df["Validation_Accuracy"],
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Number of Trees")
plt.ylabel("Score / Error")
plt.title("Random Forest: OOB Error and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 6. Bagging vs Random Forest

Bagging uses bootstrap samples and trains multiple decision trees.  
Random Forest additionally performs **feature subsampling** at each split.

Here, Bagging is compared with Random Forest using the same placement data.


In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    max_samples=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging.fit(X_train_imp, y_train)
bagging_pred = bagging.predict(X_val_imp)

bagging_acc = accuracy_score(y_val, bagging_pred)

rf = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_imp, y_train)
rf_pred = rf.predict(X_val_imp)

rf_acc = accuracy_score(y_val, rf_pred)

comparison = pd.DataFrame({
    "Model": ["Bagging", "Random Forest"],
    "Validation_Accuracy": [bagging_acc, rf_acc]
})

print(comparison)
print("\nRandom Forest OOB Score:", rf.oob_score_)
print("Random Forest OOB Error:", 1 - rf.oob_score_)


## 7. Feature Subsampling

Random Forests use a random subset of features at each tree split.  
This experiment compares different `max_features` settings.

- `sqrt`: square root of the total number of features.
- `log2`: logarithm base 2 of the total number of features.
- `None`: all features are considered at each split.


In [ ]:
feature_settings = ["sqrt", "log2", None]
feature_results = []

for setting in feature_settings:
    rf_fs = RandomForestClassifier(
        n_estimators=100,
        max_features=setting,
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )

    rf_fs.fit(X_train_imp, y_train)

    pred = rf_fs.predict(X_val_imp)

    feature_results.append({
        "max_features": str(setting),
        "OOB_Score": rf_fs.oob_score_,
        "OOB_Error": 1 - rf_fs.oob_score_,
        "Validation_Accuracy": accuracy_score(y_val, pred)
    })

feature_results_df = pd.DataFrame(feature_results)
feature_results_df


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    feature_results_df["max_features"],
    feature_results_df["Validation_Accuracy"],
    marker="o"
)

plt.xlabel("max_features")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Feature Subsampling")
plt.grid(True)
plt.show()


## 8. Feature Importance

Random Forest provides a feature importance value for each input feature.  
Higher values indicate that the feature contributed more to the model's decisions.


In [ ]:
best_feature_model = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

best_feature_model.fit(X_train_imp, y_train)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_feature_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance_df.head(15))


In [ ]:
top_n = 15

plt.figure(figsize=(9, 6))

plt.barh(
    importance_df["Feature"].head(top_n)[::-1],
    importance_df["Importance"].head(top_n)[::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Random Forest Feature Importances")
plt.show()


## 9. Result and Conclusion

### Result
- Random Forest performance was evaluated using validation accuracy.
- OOB score and OOB error were calculated for different numbers of trees.
- Bagging and Random Forest were compared.
- Different feature-subsampling strategies were evaluated using `max_features`.
- Feature importance was extracted from the Random Forest model.

### Conclusion
Random Forest combines bootstrap sampling with random feature subsampling.  
OOB error gives an internal estimate of model performance without needing a separate test set for that estimate. Feature subsampling helps make the trees less correlated, which is one of the main ideas behind Random Forests.
